# Boston Lease Checker: §5.5 evaluation

Runs the app's own pipeline (`leasecheck.analyzer.analyze` and the chat prompt) on test set v2, then the 10-seed comparison of retrieval vs the model's own knowledge. The results of record and their discussion are in `docs/evaluation.md`.

The three real leases hold personal data and are not in the repository. Upload the team's `private/` folder in step 3 to run them; without it the notebook runs the public inputs only.

## 1. Clone the repository and install

In [ ]:
# --- Erdan: clone the branch and install ---
BRANCH = 'eval/experiments-2'
!git clone -q -b $BRANCH https://github.com/SumanBadakada/IS883_bostonleasechecker.git
%cd IS883_bostonleasechecker
!pip -q install -r requirements.txt

## 2. Gemini API key

Add `GEMINI_API_KEY` in Colab's **Secrets** panel (the key icon on the left). It is never printed or saved.

In [ ]:
# --- Erdan: the Gemini key from Colab secrets ---
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

## 3. (Optional) the private real leases

Upload `private.zip` (the team's `eval/test_set/private/` folder with its `gold.json`). The hash check proves the labels are the ones committed before the run.

In [ ]:
# --- Erdan: upload the private leases and check their hash ---
from google.colab import files
import zipfile, hashlib, pathlib
for name in files.upload():
    zipfile.ZipFile(name).extractall('eval/test_set/')
digest = hashlib.sha256(pathlib.Path('eval/test_set/private/gold.json').read_bytes()).hexdigest()
print('labels unchanged since the commit:', digest == open('eval/test_set_v2/PRIVATE_GOLD.sha256').read().split()[0])
!python -m eval.build_test_set_v2   # rebuilds the inputs derived from the private lease

## 4. Every input once, with every deterministic check (about 80 requests, 10–15 minutes)

In [ ]:
# --- Erdan: main run ---
!python -m eval.run_v2 main

## 5. Retrieval vs own knowledge, seeds 1–10 (about 360 requests, about an hour)

Saved runs are skipped, so rerun the cell if the free-tier daily quota stops it.

In [ ]:
# --- Erdan: retrieval vs own knowledge ---
!python -m eval.run_v2 compare

## 5b. Tuning: 16 one-change versions on the five real leases, seeds 1–5 (about 1,200 requests)

Needs the private leases (step 3) and, in practice, a paid-tier key: the free tier's 500 requests a day would take three days. Saved runs are skipped, so rerun the cell to resume.

In [ ]:
# --- Erdan: tuning run and its summary ---
!python -m eval.tune run --seeds 5
import json, pandas as pd
from leasecheck import config
t = json.load(open(f'eval/results/v2/{config.MODEL}/tune_summary.json'))
pd.DataFrame({k: {'recall': v['recall']['mean'], 'recall_sd': v['recall']['sd'], 'precision': (v['precision'] or {}).get('mean'),
                  'output_tokens_per_lease': v['output_tokens_per_lease'], 'cost_per_1000': v['cost_per_1000_leases_usd']}
              for k, v in t.items()}).T.round(3)

## 6. Results

In [ ]:
# --- Erdan: results of the main run ---
import json, pandas as pd
from leasecheck import config
OUT = f'eval/results/v2/{config.MODEL}'   # one folder per model
s = json.load(open(f'{OUT}/main.json'))['summary']
print('checks passed:', s['checks_passed'], '/', s['checks_total'])
for part in ('real_leases', 'real_and_altered'):
    p = s[part]
    print(part, 'recall', p.get('recall_possibly_unlawful'), 'precision', p.get('precision_possibly_unlawful'))
pd.DataFrame(s['tokens_by_stage']).T

In [ ]:
# --- Erdan: results of the comparison ---
import json, pandas as pd
from leasecheck import config
OUT = f'eval/results/v2/{config.MODEL}'
c = json.load(open(f'{OUT}/compare_summary.json'))
pd.DataFrame({v: {'recall mean': x['recall']['mean'], 'recall min': x['recall']['min'], 'recall max': x['recall']['max'],
                   'recall SD': x['recall']['sd'], 'precision mean': x['precision']['mean'],
                   'input tokens / lease': x['input_tokens_per_lease'], 'cost per 1,000 leases (USD)': x['cost_per_1000_leases_usd']}
              for v, x in c.items()}).T.round(3)